Bioinformatics: AlphaFold, Invariant Point Attention, Triangle Attention, and SELFIES Chemistry

Deep learning architectures and loss functions for structural biology and molecular chemistry:
1. Triangle Multiplicative Update: Enforcing triangular consistency across residue pair representations.
2. Frame Aligned Point Error (FAPE): Local coordinate frame alignment loss invariant to global rigid motions.
3. Invariant Point Attention (IPA): SE(3)-equivariant geometric attention with 3D point Euclidean distance penalties.
4. Molecular Valence Penalty: Enforcing chemical octet rules over continuous bond adjacency matrices.

## 2. Mathematical Formulations & Theoretical Foundations

- **Triangle Multiplicative Update (Outgoing)**:
  $$S_{ij} = \sum_{k=1}^N (z_{ik} W_a + b_a) \odot (z_{jk} W_b + b_b), \quad z'_{ij} = (\sigma(z_{ij} W_g + b_g) \odot S_{ij}) W_o + b_o$$
- **Frame Aligned Point Error (FAPE)**:
  $$\mathcal{L}_{\text{FAPE}} = \frac{1}{N^2} \sum_{i, j} \min\left( \| R_{x, i}^T (\vec{x}_j - \vec{t}_{x, i}) - R_{y, i}^T (\vec{y}_j - \vec{t}_{y, i}) \|_2, d_{\text{clamp}} \right)$$
- **IPA 3D Distance Penalty**:
  $$\text{Penalty}_{ij}^h = -\frac{\gamma_h}{2} \sum_{p=1}^P \|\vec{q}_{i, h, p} - \vec{k}_{j, h, p}\|_2^2$$
- **Molecular Valence Penalty**:
  $$\mathcal{L}_{\text{valence}} = \sum_{i} \max\left( \sum_{j \ne i} \sigma(M_{ij}) - V_{\max, i}, 0 \right)^2$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "14_bioinformatics_alphafold_ipa_triangle_attention_and_selfies_chemistry.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. AlphaFold Triangle Multiplicative Update
def triangle_multiplicative_update(
    z: torch.Tensor,
    w_a: torch.Tensor,
    b_a: torch.Tensor,
    w_b: torch.Tensor,
    b_b: torch.Tensor,
    w_g: torch.Tensor,
    b_g: torch.Tensor,
    w_o: torch.Tensor,
    b_o: torch.Tensor,
) -> torch.Tensor:
    a = z @ w_a + b_a
    b = z @ w_b + b_b
    g = torch.sigmoid(z @ w_g + b_g)
    s = torch.einsum("bikc, bjkc -> bijc", a, b)
    return (g * s) @ w_o + b_o

tri_calc = triangle_multiplicative_update(
    golden["z"],
    golden["w_a"],
    golden["b_a"],
    golden["w_b"],
    golden["b_b"],
    golden["w_g"],
    golden["b_g"],
    golden["w_o"],
    golden["b_o"],
)
diff_tri = torch.max(torch.abs(tri_calc - golden["tri_out_ref"])).item()
print(f"Triangle Multiplicative Update max abs error: {diff_tri:.2e}")
assert diff_tri < 1e-5

In [ ]:
# 2. AlphaFold Frame Aligned Point Error (FAPE) Loss
def fape_loss(
    x: torch.Tensor,
    y: torch.Tensor,
    r_x: torch.Tensor,
    t_x: torch.Tensor,
    r_y: torch.Tensor,
    t_y: torch.Tensor,
    d_clamp: float = 10.0,
    eps: float = 1e-12,
) -> torch.Tensor:
    diff_x = x.unsqueeze(1) - t_x.unsqueeze(2)
    local_x = torch.einsum("bijc, biac -> bija", diff_x, r_x)

    diff_y = y.unsqueeze(1) - t_y.unsqueeze(2)
    local_y = torch.einsum("bijc, biac -> bija", diff_y, r_y)

    dist = torch.sqrt(torch.sum((local_x - local_y) ** 2, dim=-1) + eps)
    clamped_dist = torch.clamp(dist, max=d_clamp)
    return clamped_dist.mean()

fape_calc = fape_loss(
    golden["x"],
    golden["y"],
    golden["r_x"],
    golden["t_x"],
    golden["r_y"],
    golden["t_y"],
    d_clamp=10.0,
)
diff_fape = torch.abs(fape_calc - golden["fape_loss_ref"]).item()
print(f"FAPE Loss max abs error: {diff_fape:.2e}")
assert diff_fape < 1e-5

In [ ]:
# 3. AlphaFold Invariant Point Attention (IPA) Point Distance Penalty
def ipa_point_distance_penalty(q_pts: torch.Tensor, k_pts: torch.Tensor, gamma: torch.Tensor) -> torch.Tensor:
    diff = q_pts.unsqueeze(2) - k_pts.unsqueeze(1)
    sq_dist = torch.sum(diff ** 2, dim=(-2, -1))
    return -0.5 * gamma.view(1, 1, 1, -1) * sq_dist

ipa_calc = ipa_point_distance_penalty(golden["q_pts"], golden["k_pts"], golden["gamma"])
diff_ipa = torch.max(torch.abs(ipa_calc - golden["ipa_penalty_ref"])).item()
print(f"IPA Point Distance Penalty max abs error: {diff_ipa:.2e}")
assert diff_ipa < 1e-5

In [ ]:
# 4. Molecular Valence Penalty & Summary Table
def molecular_valence_loss(bond_logits: torch.Tensor, max_valence: torch.Tensor) -> torch.Tensor:
    b, n, _ = bond_logits.shape
    probs = torch.sigmoid(bond_logits)
    mask = 1.0 - torch.eye(n, device=bond_logits.device).unsqueeze(0)
    masked_probs = probs * mask
    valences = torch.sum(masked_probs, dim=-1)
    excess = F.relu(valences - max_valence)
    return torch.mean(torch.sum(excess ** 2, dim=-1))

val_calc = molecular_valence_loss(golden["bond_logits"], golden["max_valence"])
diff_val = torch.abs(val_calc - golden["valence_loss_ref"]).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'Triangle Multiplicative Update':<32} | {diff_tri:<15.2e} | PASSED")
print(f"{'FAPE Structural Loss':<32} | {diff_fape:<15.2e} | PASSED")
print(f"{'IPA Point Distance Penalty':<32} | {diff_ipa:<15.2e} | PASSED")
print(f"{'Molecular Valence Penalty':<32} | {diff_val:<15.2e} | PASSED")
print("=" * 65)
assert diff_val < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] Triangle multiplicative updates propagate pairwise geometric constraints across residue graphs.
- [x] FAPE loss provides coordinate-frame alignment that evaluates local conformational fidelity without hinge sensitivity.
- [x] Invariant Point Attention computes SE(3)-equivariant geometric representations using local coordinate systems.
- [x] Valence regularization enforces chemical octet validity during de novo molecular graph generation.